# Produto da dissertação — organização dos CSVs de ocorrências

**Crédito rural no Cerrado, 2021–2025 | versão 1.0**

Este notebook transforma resultados existentes em um pacote tabular documentado. Não refaz cruzamentos espaciais, não altera os arquivos originais e não realiza verificação administrativa ou jurídica.

**Entregas:** sete CSVs leves, as duas visões de embargos (931 e 37 registros), dicionário de colunas, relação das definições pendentes, inventário das origens, relatório de conferências, guia de interpretação e ZIP. Todas as linhas são preservadas; apenas colunas de geometria são omitidas das cópias leves.

## Como executar no Colab
1. Abra este `.ipynb` em https://colab.research.google.com usando **Arquivo → Fazer upload de notebook**.
2. A pasta **Envio** deve estar acessível no seu Drive. Se estiver em “Compartilhados comigo”, adicione um atalho em “Meu Drive”.
3. Execute a montagem do Drive e confira `RAIZ` na configuração. Se necessário, use o painel de arquivos do Colab para copiar o caminho correto.
4. Execute as células na ordem. A conferência inicial mostra exatamente os arquivos selecionados. Ajuste apenas caminhos que não correspondam à sua organização.
5. Ao terminar, veja `status.json`, `validacao.csv` e `definicoes_pendentes.csv` na pasta de saída. O ZIP só é gerado quando as conferências implementadas passam.

**Tempo de execução:** as bases de florestas públicas e UCs têm, juntas, mais de 33 GB. A leitura por blocos reduz o uso de memória, mas precisa percorrer os CSVs inteiros. Reserve tempo de execução no Colab; o processamento não é instantâneo. Se a sessão for interrompida, a nova execução cria outra pasta e recomeça a leitura. Não apaga tentativas anteriores.

**Seleção das versões:** os caminhos foram configurados com base nos arquivos observados em `Envio/Arquivos gerados e relatorios`. Não há seleção automática do arquivo mais recente. As contagens servem de controle, mas não substituem a confirmação metodológica da versão definitiva.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 1. Configuração

`RAIZ` deve apontar para **Arquivos gerados e relatorios**. Os arquivos de saída serão criados em uma nova pasta com data e hora, dentro de `Produto_dissertacao_Cerrado`.

Não altere as contagens esperadas apenas para fazer a conferência passar. Uma divergência precisa ser investigada. Os IDs Drive documentam a seleção original; a leitura é feita pelos caminhos montados, e o notebook não verifica novamente os IDs pela API.


In [2]:
from pathlib import Path
import json

RAIZ = Path('/content/drive/MyDrive/Envio/Arquivos gerados e relatorios')
SAIDA_RAIZ = Path('/content/drive/MyDrive/Produto_dissertacao_Cerrado')
TAMANHO_BLOCO = 100  # reduza se houver limitação de memória

# Opcional: apenas se um arquivo tiver sido movido, informe seu caminho completo.
# Exemplo: CAMINHOS_MANUAIS = {'embargos_prioritarios': '/content/drive/MyDrive/.../casos_prioritarios_desde_2024.csv'}
CAMINHOS_MANUAIS = {}

CONFIGURACAO = json.loads(r'''{
  "terras_indigenas": {
    "relativo": "intersecoes/terras indígenas 2/resultados_por_id_gleba2/versao_2/bases_analiticas/base_completa_sem_deduplicacao.csv",
    "id": "1ByUyBx6I8FAfi8cB-A9egaVGhRMpBHwL",
    "esperado": {
      "registros": 660,
      "glebas": 554,
      "operacoes": 476,
      "unidades": 43
    },
    "criterio": "Base completa da versão 2 observada na pasta; não usar partes municipais nem escolher pela data mais recente."
  },
  "quilombolas": {
    "relativo": "intersecoes/quilombolas/resultados_por_id_gleba2/versao_2/bases_analiticas/base_completa_sem_deduplicacao.csv",
    "id": "1EgzYfTaSmBrDsVBNmdHLyr219r0xs0nG",
    "esperado": {
      "registros": 368,
      "glebas": 300,
      "operacoes": 234,
      "unidades": 43
    },
    "criterio": "Base completa da versão 2 observada na pasta; não usar partes municipais nem escolher pela data mais recente."
  },
  "florestas_publicas": {
    "relativo": "intersecoes/florestas publicas/resultados_por_id_gleba2/versao_2/bases_analiticas/base_completa_sem_deduplicacao.csv",
    "id": "1qhQXQ7twMBg9t5YdoQM0fo-F29wFOd04",
    "esperado": {
      "registros": 33984,
      "glebas": 25284,
      "operacoes": 25864,
      "unidades": 1766
    },
    "criterio": "Base completa da versão 2 observada na pasta; não usar partes municipais nem escolher pela data mais recente."
  },
  "assentamentos": {
    "relativo": "intersecoes/assentamentos/resultados_por_id_gleba2/versao_2/bases_analiticas/base_completa_sem_deduplicacao.csv",
    "id": "1dPBBKxB9LHDBycmjSMWnLeeKzziaa5l4",
    "esperado": {
      "registros": 33044,
      "glebas": 24615,
      "operacoes": 26281,
      "unidades": 1568
    },
    "criterio": "Base completa da versão 2 observada na pasta; não usar partes municipais nem escolher pela data mais recente."
  },
  "embargos": {
    "relativo": "intersecoes/embargos/resultados_por_id_gleba2/versao_2/bases_analiticas/base_completa_sem_deduplicacao.csv",
    "id": "1iK_PK-8ksg2fmTMTR6_f6Vn5In3QJdTZ",
    "esperado": {
      "registros": 931,
      "glebas": 626,
      "operacoes": 570,
      "unidades": 388
    },
    "criterio": "Base completa da versão 2 observada na pasta; não usar partes municipais nem escolher pela data mais recente."
  },
  "unidades_conservacao": {
    "relativo": "intersecoes/unidades de conservacao/resultados_por_id_gleba2/versao_2/bases_analiticas/base_completa_sem_deduplicacao.csv",
    "id": "1GWw06MzWnz66l4x5aPX9T4l91vcoRxmc",
    "esperado": {
      "registros": 18617,
      "glebas": 14621,
      "operacoes": 11757,
      "unidades": 208
    },
    "criterio": "Base completa da versão 2 observada na pasta; não usar partes municipais nem escolher pela data mais recente."
  },
  "car_suspenso": {
    "relativo": "intersecoes/sicar_suspensos/resultados_por_id_gleba2/versao_2/bases_analiticas/base_completa_sem_deduplicacao.csv",
    "id": "1rA9V5dpsAts1U_HOrUndga-Sg7yetpDl",
    "esperado": {
      "registros": 530,
      "glebas": 377,
      "operacoes": 368,
      "unidades": 214
    },
    "criterio": "Base completa da versão 2 observada na pasta; não usar partes municipais nem escolher pela data mais recente."
  },
  "embargos_auditados": {
    "relativo": "complementos_embargos_5_5_5/20260918_142715_010681/tabelas/931_linhas_auditadas.csv",
    "id": "1CZ-FeZDxUXK0Xao1m2Ei_czqTqMNsXQo",
    "esperado": {
      "registros": 931,
      "glebas": 626,
      "operacoes": 570,
      "unidades": 388
    },
    "criterio": "Arquivo complementar de setembro já conferido; visão dos embargos, não somar às sete camadas."
  },
  "embargos_prioritarios": {
    "relativo": "complementos_embargos_5_5_5/20260918_142715_010681/tabelas/casos_prioritarios_desde_2024.csv",
    "id": "1ECC6EtABqMk50-MHZdVHKnagLeTEa5-r",
    "esperado": {
      "registros": 37,
      "glebas": 34,
      "operacoes": 31,
      "unidades": 25
    },
    "criterio": "Arquivo complementar de setembro já conferido; visão dos embargos, não somar às sete camadas."
  }
}''')


## 2. Funções de processamento e documentação

O código preserva identificadores e atributos como texto, sem arredondar valores, eliminar repetições ou converter automaticamente datas. As verificações temporais usam cópias das datas em formato ISO. Campos cuja definição não foi confirmada permanecem no CSV e são sinalizados no dicionário.


In [3]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv, hashlib, json, re, zipfile, traceback
import pandas as pd

VERSAO_NOTEBOOK = '1.0'

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()

def salvar_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')

def salvar_csv(path, linhas):
    df = pd.DataFrame(linhas)
    df.to_csv(path, sep=';', index=False, encoding='utf-8-sig', lineterminator='\n')

def assinatura(path):
    s = Path(path).stat()
    return {'tamanho_bytes': s.st_size, 'mtime_ns': s.st_mtime_ns}

def cabecalho(path):
    # Detecta somente entre os delimitadores esperados. Não pula linhas inválidas.
    with Path(path).open('r', encoding='utf-8-sig', newline='') as f:
        primeira = f.readline()
    candidatos = {sep: next(csv.reader([primeira], delimiter=sep)) for sep in [';', ',', '\t']}
    sep = max(candidatos, key=lambda x: len(candidatos[x]))
    cols = candidatos[sep]
    if len(cols) < 2 or len(set(cols)) != len(cols) or any(not c.strip() for c in cols):
        raise ValueError(f'Cabeçalho inválido ou duplicado: {path}')
    return sep, cols

def geometrica(coluna):
    c = coluna.upper()
    return bool(re.search(r'WKT|GEOMETRIA|GEOMETRY|GEOJSON|WKB', c) or c in {'GEOM', 'SHAPE'})

ALIASES = {
    'glebas': ['_ID_GLEBA2', 'ID_GLEBA2', 'id_gleba2', 'gleba'],
    'operacoes': ['_REF_BACEN', 'REF_BACEN', 'ref'],
    'unidades': ['_UNIDADE_ID', 'unidade'],
}

DEFINICOES = {
    'ID_GLEBA': ('Identificador de gleba na base de origem.', 'identificador', 'não aplicável'),
    'ID_GLEBA2': ('Identificador de representações com WKT idêntico; não equivale a imóvel ou operação.', 'identificador', 'não aplicável'),
    '_ID_GLEBA2': ('Identificador analítico derivado de ID_GLEBA2.', 'identificador', 'não aplicável'),
    'REF_BACEN': ('Identificador da operação no SICOR; pode aparecer em várias linhas.', 'identificador', 'não aplicável'),
    '_REF_BACEN': ('Identificador analítico da operação, derivado de REF_BACEN.', 'identificador', 'não aplicável'),
    'NU_ORDEM_OPERACAO': ('Ordem da operação; distinguir de NU_ORDEM_GLEBA.', 'identificador', 'não aplicável'),
    'NU_ORDEM_GLEBA': ('Ordem da gleba informada na base SICOR.', 'identificador', 'não aplicável'),
    'CHAVE_ORDEM_CREDITO': ('Chave da ordem de crédito na base de origem.', 'identificador', 'não aplicável'),
    'DT_EMISSAO': ('Data de emissão da operação utilizada na comparação temporal.', 'data de origem', 'data'),
    'ANO_BASE': ('Ano do arquivo territorial de origem; não substituir pela data de emissão.', 'ano', 'ano'),
    'ANO_OPERACAO': ('Ano da operação informado na base analítica.', 'ano', 'ano'),
    'ANO_ARQUIVO_OPERACAO': ('Ano do arquivo de operações utilizado no vínculo.', 'ano', 'ano'),
    'VL_PARC_CREDITO': ('Valor da parcela de crédito da origem; pode repetir nas linhas de interseção.', 'numérico de origem', 'R$'),
    'AREA_INTERSECAO_HA': ('Área da interseção calculada anteriormente; não recalculada neste notebook.', 'numérico de origem', 'ha'),
    'AREA_GLEBA_HA_CALCULADA': ('Área calculada da geometria da gleba na origem.', 'numérico de origem', 'ha'),
    'PERCENTUAL_GLEBA_INTERSECTADA': ('Percentual da área da gleba que intersecta a unidade da camada.', 'numérico de origem', '%'),
    '_ID_LINHA_ORIGINAL': ('Identificador da linha atribuído no processamento anterior.', 'identificador', 'não aplicável'),
    '_UNIDADE_ID': ('Identificador analítico da unidade territorial; interpretar no âmbito de cada camada.', 'identificador', 'não aplicável'),
    '_UNIDADE_NOME': ('Nome da unidade territorial preservado da origem.', 'texto', 'não aplicável'),
    '_VALOR_R': ('Valor financeiro analítico da linha; não representa crédito exclusivo da área intersectada.', 'numérico de origem', 'R$'),
    '_AREA_INTERSECAO_CSV_HA': ('Área da interseção recuperada do CSV de origem.', 'numérico de origem', 'ha'),
    '_AREA_GLEBA_CSV_HA': ('Área da gleba recuperada do CSV de origem.', 'numérico de origem', 'ha'),
    '_AREA_TOTAL_UNIDADE_HA': ('Área total da unidade territorial conforme a base analítica.', 'numérico de origem', 'ha'),
    '_PERCENTUAL_GLEBA': ('Percentual da gleba intersectado conforme a base analítica.', 'numérico de origem', '%'),
    '_PROGRAMA': ('Programa de crédito informado na base analítica.', 'texto', 'não aplicável'),
    '_TIPO_PRODUTOR': ('Categoria de produtor utilizada no processamento anterior.', 'texto', 'não aplicável'),
    'UF_OPERACAO': ('UF da operação conforme o arquivo de origem; não é inferência espacial deste notebook.', 'texto', 'UF'),
    'TIPO_IMPEDIMENTO': ('Rótulo legado da camada na origem; não confirma impedimento jurídico.', 'texto', 'não aplicável'),
    'STATUS_IMPEDIMENTO': ('Rótulo legado da condição na origem; não é decisão administrativa.', 'texto', 'não aplicável'),
    'linha': ('Identificador da linha na auditoria temporal de origem.', 'identificador', 'não aplicável'),
    'gleba': ('Identificador da gleba na auditoria temporal (ID_GLEBA2).', 'identificador', 'não aplicável'),
    'ref': ('REF_BACEN da operação na auditoria temporal.', 'identificador', 'não aplicável'),
    'unidade': ('Identificador do polígono de embargo na auditoria temporal.', 'identificador', 'não aplicável'),
    'nome': ('Nome ou rótulo do polígono de embargo.', 'texto', 'não aplicável'),
    'termo': ('Número ou código do termo de embargo preservado como texto.', 'identificador', 'não aplicável'),
    'serie': ('Série do termo de embargo, quando disponível.', 'identificador', 'não aplicável'),
    'ano_operacao': ('Ano da operação na auditoria temporal.', 'ano', 'ano'),
    'categoria_produtor': ('Categoria do produtor na auditoria temporal.', 'texto', 'não aplicável'),
    'data_operacao': ('Data da operação utilizada na auditoria temporal.', 'data ISO', 'data'),
    'data_embargo': ('Data do embargo utilizada na auditoria temporal; não comprova vigência posterior.', 'data ISO', 'data'),
    'dias_entre_embargo_e_operacao': ('Diferença em dias entre a data da operação e a do embargo.', 'inteiro', 'dias'),
    'mesma_data': ('Indicador de igualdade entre as duas datas na origem.', 'booleano de origem', 'não aplicável'),
    'relacao_temporal': ('Classificação temporal produzida na origem.', 'texto', 'não aplicável'),
    'periodo_origem': ('Faixa temporal e referência normativa registradas na análise anterior; sem revisão jurídica aqui.', 'texto', 'não aplicável'),
    'classificacao': ('Classificação de triagem preservada; não constitui conclusão de irregularidade.', 'texto', 'não aplicável'),
    'prioritario': ('Indicador do recorte prioritário na análise anterior.', 'booleano de origem', 'não aplicável'),
    'vigencia_confirmada_origem': ('Indicador de confirmação na origem. False significa não confirmada, e não embargo sem vigência.', 'booleano de origem', 'não aplicável'),
    'encerramento_origem': ('Informação de encerramento disponível na origem; campo vazio não prova ausência de encerramento.', 'texto de origem', 'não aplicável'),
    'observacao_vigencia': ('Observação sobre a possibilidade de confirmar a vigência do embargo.', 'texto', 'não aplicável'),
    'valor': ('Valor financeiro associado à linha; pode repetir entre ocorrências.', 'numérico de origem', 'R$'),
    'area_ha': ('Área de interseção da linha na auditoria temporal; não representa união espacial.', 'numérico de origem', 'ha'),
    'produto_id_registro': ('Chave da exportação: conjunto + posição do registro CSV; não é chave administrativa.', 'identificador', 'não aplicável'),
    'produto_registro_origem': ('Posição de dados no CSV, iniciada em 1, excluindo o cabeçalho; não é linha física.', 'inteiro', 'registro'),
    'produto_conjunto': ('Nome do conjunto exportado.', 'texto', 'não aplicável'),
    'produto_escopo': ('Natureza de triagem dos registros.', 'texto', 'não aplicável'),
}

def documentar(conjunto, colunas, ausencias, n, descartadas):
    linhas = []
    for c in colunas + descartadas:
        if c in descartadas:
            desc, tipo, unidade, estado = ('Geometria omitida da cópia leve; permanece no arquivo original.', 'geometria', 'origem', 'omitida da cópia leve')
        elif c in DEFINICOES:
            desc, tipo, unidade = DEFINICOES[c]
            estado = 'descrição fornecida pelo notebook; revisar com a metodologia'
        else:
            desc, tipo, unidade, estado = ('Atributo preservado literalmente. Consultar o dicionário da fonte antes de interpretar.', 'texto de origem', 'não confirmada', 'definição pendente')
        linhas.append({'conjunto': conjunto, 'coluna': c, 'incluida_no_csv_leve': c not in descartadas,
                       'descricao': desc, 'tipo_logico': tipo, 'unidade': unidade, 'situacao_definicao': estado,
                       'vazios': ausencias.get(c, ''), 'registros': n,
                       'armazenamento': 'texto; valores e separadores decimais preservados'})
    return linhas

def exportar_arquivo(chave, cfg, origem, pasta, bloco=100):
    antes = assinatura(origem)
    sep, todas = cabecalho(origem)
    descartadas = [c for c in todas if geometrica(c)]
    manter = [c for c in todas if c not in descartadas]
    novas = ['produto_id_registro', 'produto_registro_origem', 'produto_conjunto', 'produto_escopo']
    if set(novas) & set(todas):
        raise ValueError('A entrada parece já ser uma exportação deste produto.')
    chaves = {k: [c for c in alternativas if c in manter] for k, alternativas in ALIASES.items()}
    if any(not v for v in chaves.values()):
        raise ValueError(f'{chave}: faltam identificadores esperados: {chaves}')
    destino = pasta / (chave + '.csv')
    temporario = destino.with_suffix('.csv.part')
    unicos = {k: set() for k in chaves}
    ausencias, conflitos = Counter(), Counter()
    n, proximo_aviso = 0, 1000
    # usecols evita carregar as geometrias no DataFrame. O CSV ainda precisa ser lido integralmente.
    with temporario.open('w', encoding='utf-8-sig', newline='') as saida:
        leitor = pd.read_csv(origem, sep=sep, encoding='utf-8-sig', dtype=str,
                             keep_default_na=False, na_filter=False, usecols=manter,
                             chunksize=bloco, engine='c', on_bad_lines='error')
        for df in leitor:
            for k, cs in chaves.items():
                principal = df[cs[0]].str.strip()
                unicos[k].update(v for v in principal if v)
                conflitos['vazios_' + k] += int((principal == '').sum())
                for outra in cs[1:]:
                    conflitos['aliases_' + k] += int((principal != df[outra].str.strip()).sum())
            for c in manter:
                ausencias[c] += int(df[c].str.strip().eq('').sum())
            # Detecta geometria com nome inesperado, evitando entregá-la silenciosamente.
            for c in manter:
                if df[c].str.match(r'^\s*(?:SRID=\d+;)?(?:MULTI)?(?:POLYGON|LINESTRING|POINT|GEOMETRYCOLLECTION)\s*[Z M]*\s*\(', case=False).any():
                    raise ValueError(f'Geometria em coluna não reconhecida: {c}. Acrescente à função geometrica antes de executar novamente.')
            posicoes = range(n + 1, n + len(df) + 1)
            df.insert(0, 'produto_id_registro', [f'{chave}:{i:08d}' for i in posicoes])
            df.insert(1, 'produto_registro_origem', list(posicoes))
            df.insert(2, 'produto_conjunto', chave)
            df.insert(3, 'produto_escopo', 'Ocorrência para triagem; verificação administrativa não executada')
            df.to_csv(saida, sep=';', index=False, header=(n == 0), lineterminator='\n')
            n += len(df)
            if n >= proximo_aviso:
                print(f'  {chave}: {n:,} registros lidos', flush=True)
                proximo_aviso = n + 1000
    if antes != assinatura(origem):
        raise RuntimeError('Arquivo de entrada alterado durante a leitura; resultado não finalizado.')
    if n == 0:
        raise ValueError('Arquivo sem registros.')
    temporario.replace(destino)
    controles = []
    calculados = {'registros': n, **{k: len(v) for k, v in unicos.items()}}
    for k, v in cfg['esperado'].items():
        controles.append({'conjunto': chave, 'teste': k, 'esperado': v, 'calculado': calculados[k], 'confere': v == calculados[k]})
    for k, v in conflitos.items():
        controles.append({'conjunto': chave, 'teste': k, 'esperado': 0, 'calculado': v, 'confere': v == 0})
    inventario = {'conjunto': chave, 'arquivo_origem': str(origem), 'drive_id_origem': cfg['id'],
                  'criterio_selecao': cfg['criterio'], 'assinatura_origem': antes,
                  'arquivo_saida': destino.name, 'sha256_saida': sha256(destino),
                  'tamanho_saida_bytes': destino.stat().st_size, 'contagens': calculados,
                  'colunas_omitidas': descartadas, 'delimitador_origem': sep,
                  'hash_integral_origem': 'não calculado; origem identificada por caminho, ID Drive, tamanho e mtime',
                  'preservacao': 'Todas as linhas e todos os atributos não geométricos; sem deduplicação'}
    return inventario, controles, documentar(chave, novas + manter, ausencias, n, descartadas)

def conferir_temporal(pasta):
    geral = pd.read_csv(pasta / 'embargos_auditados.csv', sep=';', dtype=str, keep_default_na=False)
    prior = pd.read_csv(pasta / 'embargos_prioritarios.csv', sep=';', dtype=str, keep_default_na=False)
    controles = []
    def teste(nome, esperado, valor):
        controles.append({'conjunto': 'conferencia_temporal', 'teste': nome, 'esperado': esperado, 'calculado': valor, 'confere': esperado == valor})
    obrigatorias = ['linha', 'data_operacao', 'data_embargo', 'prioritario']
    for df in (geral, prior):
        faltam = set(obrigatorias) - set(df.columns)
        if faltam:
            raise ValueError(f'Campos temporais ausentes: {faltam}')
    op = pd.to_datetime(geral.data_operacao, format='%Y-%m-%d', errors='coerce')
    emb = pd.to_datetime(geral.data_embargo, format='%Y-%m-%d', errors='coerce')
    teste('datas_invalidas', 0, int((op.isna() | emb.isna()).sum()))
    teste('embargo_anterior', 400, int((emb < op).sum()))
    teste('embargo_posterior', 531, int((emb > op).sum()))
    teste('mesma_data', 0, int((emb == op).sum()))
    # Reproduz apenas o recorte já utilizado no trabalho; não avalia vigência ou exceções.
    filtro = (emb < op) & (op >= pd.Timestamp('2024-01-02'))
    teste('recorte_temporal', 37, int(filtro.sum()))
    teste('linhas_duplicadas_na_auditoria', 0, int(geral.linha.duplicated().sum()))
    teste('linhas_duplicadas_no_recorte', 0, int(prior.linha.duplicated().sum()))
    teste('mesmos_identificadores_do_recorte', True, set(geral.loc[filtro, 'linha']) == set(prior.linha))
    rotulos = geral.prioritario.str.casefold()
    teste('rotulos_prioritario_invalidos', 0, int((~rotulos.isin(['true', 'false'])).sum()))
    teste('rotulo_prioritario_coerente', True, bool(rotulos.eq('true').equals(filtro)))
    teste('rotulos_no_recorte', True, bool(prior.prioritario.str.casefold().eq('true').all()))
    comuns = [c for c in geral.columns if not c.startswith('produto_') and c in prior.columns]
    if not geral.linha.duplicated().any() and not prior.linha.duplicated().any():
        a = geral.loc[filtro, comuns].set_index('linha').sort_index()
        b = prior[comuns].set_index('linha').sort_index()
        teste('atributos_do_recorte_iguais_a_auditoria', True, a.equals(b))
    return controles

GUIA = '''# Guia de utilização do produto

## Conteúdo e alcance
As tabelas descrevem ocorrências de interseção territorial no conjunto público analisado.
Os arquivos das sete camadas preservam as linhas das bases completas da versão 2.
Os arquivos embargos_auditados e embargos_prioritarios são visões complementares dos embargos,
não novas ocorrências a adicionar ao conjunto. O recorte prioritário reproduz o filtro temporal
já empregado na dissertação: embargo anterior e emissão a partir de 02/01/2024.
Essa data é parâmetro do recorte anterior, não validação jurídica feita por este notebook.

## Como localizar um caso
Use produto_id_registro para rastrear a exportação e produto_registro_origem para recuperar
a posição no CSV original. Use REF_BACEN (ou ref), ordem da operação quando disponível,
ID_GLEBA2 (ou gleba) e identificador da unidade para a verificação administrativa.
Uma linha não equivale necessariamente a uma operação, imóvel ou beneficiário distinto.
Os códigos são preservados como texto, inclusive zeros à esquerda. Ao abrir no Excel,
importe por Dados > De Texto/CSV, selecione UTF-8, delimitador ponto e vírgula e mantenha
identificadores como texto. Datas e números mantêm a grafia da origem. Existem bases com
vírgula decimal e outras com ponto decimal: não substitua separadores indiscriminadamente.

## Interpretação
- Interseção não confirma desconformidade jurídica nem responsabilidade individual.
- Embargo anterior não confirma que a restrição permanecia vigente na emissão do crédito.
- False em vigencia_confirmada_origem significa ausência de confirmação na análise,
  não confirmação de ausência de vigência.
- CAR suspenso retrata a situação da base utilizada, sem reconstruir sua situação histórica.
- Assentamentos são uma camada de contexto territorial, não impedimento automático.
- Fase de terra indígena, situação quilombola, tipo de floresta e categoria de UC devem
  ser interpretados com seus atos e atributos de origem.
- Campos legados com a palavra impedimento são preservados por rastreabilidade e não
  constituem conclusão jurídica deste produto.

## Áreas e valores
Áreas por linha podem se repetir. A soma das áreas não corresponde à união espacial.
Valores financeiros por linha podem repetir a mesma operação/ordem e não estão rateados
pela superfície intersectada. Não os apresente como crédito irregular ou crédito exclusivo
da área. Não some os totais das sete camadas nem adicione as duas visões de embargos.
Este notebook não recalcula geometrias, área de união ou totais financeiros corrigidos.

## Documentação e conferências
inventario_origens.json registra caminhos, IDs Drive, tamanhos, datas de modificação,
critérios de seleção e hashes dos CSVs de saída. O hash integral das bases de entrada não
é calculado, para evitar uma segunda leitura de dezenas de GB.
dicionario_colunas.csv cobre todas as colunas preservadas e as geometrias omitidas.
Campos sem definição confirmada ficam explicitamente marcados como definição pendente;
essas descrições precisam ser completadas com os dicionários das fontes.
validacao.csv compara registros e identificadores distintos com as referências da dissertação
e verifica a consistência das duas visões temporais. A aprovação dessas contagens não é
auditoria de cada atributo nem demonstra, isoladamente, que a versão é definitiva.

## Consulta administrativa posterior
A confirmação documental de cada caso não foi executada por este notebook.
A identificação das operações, a vigência dos atos, o enquadramento aplicável e eventuais
exceções deverão ser examinados na etapa de verificação. Registre os resultados dessa etapa
em uma tabela separada, vinculada aos identificadores, preservando os resultados da pesquisa.
'''

def executar_produto(raiz, saida_raiz, configuracao, caminhos_manuais=None, bloco=100):
    raiz = Path(raiz)
    caminhos_manuais = caminhos_manuais or {}
    selecionados, problemas = {}, []
    for chave, cfg in configuracao.items():
        origem = Path(caminhos_manuais[chave]) if chave in caminhos_manuais else raiz / cfg['relativo']
        if not origem.is_file():
            problemas.append(f'{chave}: arquivo não encontrado em {origem}')
        else:
            selecionados[chave] = origem
    if problemas:
        raise FileNotFoundError('\n'.join(problemas) + '\nAjuste RAIZ ou CAMINHOS_MANUAIS. Nenhum arquivo de dados foi processado.')
    momento = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    pasta = Path(saida_raiz) / ('produto_' + momento)
    pasta.mkdir(parents=True, exist_ok=False)
    dados = pasta / 'dados'
    dados.mkdir()
    inventario, controles, dicionario, falhas = [], [], [], []
    print('Saída:', pasta)
    for chave, cfg in configuracao.items():
        print(f'Processando {chave} ({selecionados[chave].stat().st_size / 1e9:.2f} GB)...', flush=True)
        try:
            inv, checks, dic = exportar_arquivo(chave, cfg, selecionados[chave], dados, bloco)
            inventario.append(inv)
            controles.extend(checks)
            dicionario.extend(dic)
        except Exception as e:
            falhas.append({'conjunto': chave, 'erro': str(e)})
            controles.append({'conjunto': chave, 'teste': 'processamento', 'esperado': 'concluído', 'calculado': str(e), 'confere': False})
            print('  Falha registrada:', e)
        # Grava evidências parciais para diagnóstico caso a sessão seja interrompida.
        salvar_json(pasta / 'inventario_origens.json', inventario)
        salvar_csv(pasta / 'validacao.csv', controles)
    if (dados / 'embargos_auditados.csv').exists() and (dados / 'embargos_prioritarios.csv').exists():
        try:
            controles.extend(conferir_temporal(dados))
        except Exception as e:
            controles.append({'conjunto': 'temporal', 'teste': 'processamento', 'esperado': 'concluído', 'calculado': str(e), 'confere': False})
    salvar_csv(pasta / 'validacao.csv', controles)
    salvar_csv(pasta / 'dicionario_colunas.csv', dicionario)
    pendentes = [d for d in dicionario if d['situacao_definicao'] == 'definição pendente']
    salvar_csv(pasta / 'definicoes_pendentes.csv', pendentes or [{'conjunto': '', 'coluna': '', 'situacao_definicao': 'Nenhuma definição pendente'}])
    (pasta / 'LEIA_ME.md').write_text(GUIA, encoding='utf-8')
    divergencias = [c for c in controles if not c['confere']]
    status = {'versao_notebook': VERSAO_NOTEBOOK, 'gerado_utc': momento,
              'conjuntos_previstos': len(configuracao), 'conjuntos_exportados': len(inventario),
              'divergencias': len(divergencias), 'falhas': falhas,
              'definicoes_pendentes': len(pendentes),
              'status': 'BLOQUEADO_POR_DIVERGENCIAS' if divergencias else 'CONTAGENS_CONFERIDAS_DOCUMENTACAO_PARA_REVISAO',
              'verificacao_administrativa': 'não executada', 'intersecoes_recalculadas': False}
    salvar_json(pasta / 'status.json', status)
    salvar_json(pasta / 'configuracao.json', configuracao)
    print(json.dumps(status, ensure_ascii=False, indent=2))
    if divergencias:
        print('ZIP não gerado. Consulte validacao.csv; os resultados parciais permanecem identificados nesta pasta.')
        return pasta, None
    zip_path = pasta.with_suffix('.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for p in sorted(pasta.rglob('*')):
            if p.is_file() and not p.name.endswith('.part'):
                z.write(p, p.relative_to(pasta))
    print('Pacote gerado:', zip_path)
    print('As contagens conferem. Leia o dicionário e suas pendências antes da entrega definitiva.')
    return pasta, zip_path


## 3. Conferência dos caminhos e dos cabeçalhos

Esta célula apenas lê os cabeçalhos e metadados dos arquivos. Ela interrompe a execução se alguma entrada estiver ausente. As colunas omitidas devem ser exclusivamente geometrias. A cópia original continua sendo a referência para mapas e recálculo de áreas.


In [4]:
# if not RAIZ.is_dir():
#     raise FileNotFoundError(f'Pasta não encontrada: {RAIZ}. Ajuste RAIZ na célula 1.')
# itens = []
# for chave, cfg in CONFIGURACAO.items():
#     p = Path(CAMINHOS_MANUAIS.get(chave, RAIZ / cfg['relativo']))
#     item = {'conjunto': chave, 'encontrado': p.is_file(), 'caminho': str(p)}
#     if p.is_file():
#         sep, cols = cabecalho(p)
#         item.update({'tamanho_GB': round(p.stat().st_size / 1e9, 3),
#                      'registros_esperados': cfg['esperado']['registros'],
#                      'colunas_omitidas': ', '.join(c for c in cols if geometrica(c))})
#     itens.append(item)
# display(pd.DataFrame(itens))
# if not all(i['encontrado'] for i in itens):
#     raise FileNotFoundError('Há arquivos ausentes. Corrija RAIZ ou CAMINHOS_MANUAIS antes de continuar.')
# print('Tamanho total das entradas (GB):', round(sum(Path(i['caminho']).stat().st_size for i in itens) / 1e9, 2))


FileNotFoundError: Pasta não encontrada: /content/drive/MyDrive/Envio/Arquivos gerados e relatorios. Ajuste RAIZ na célula 1.

In [5]:
from pathlib import Path
import os
import unicodedata
import pandas as pd

# 1. Conectar o Drive, se estiver no Colab e ainda não estiver montado.
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
except ImportError:
    pass  # Fora do Colab, utiliza as pastas disponíveis no ambiente.

def normalizar_nome(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    return "".join(
        c for c in texto if not unicodedata.combining(c)
    ).casefold().strip()

# 2. Procurar possíveis localizações da pasta.
locais = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path.cwd(),
]
locais = list(dict.fromkeys(p for p in locais if p.is_dir()))

candidatas = set()
raiz_anterior = Path(RAIZ)

if raiz_anterior.is_dir():
    candidatas.add(raiz_anterior)

nome_procurado = normalizar_nome("Arquivos gerados e relatorios")

for local in locais:
    # O próprio local também pode ser a raiz dos arquivos.
    candidatas.add(local)

    for atual, subpastas, arquivos in os.walk(local):
        atual = Path(atual)
        profundidade = len(atual.relative_to(local).parts)

        # Evitar pastas ocultas e resultados de execuções anteriores.
        subpastas[:] = [
            nome for nome in subpastas
            if not nome.startswith(".")
            and not nome.startswith("produto_")
            and nome != "Produto_dissertacao_Cerrado"
        ]

        if profundidade >= 8:
            subpastas[:] = []

        if (
            normalizar_nome(atual.name) == nome_procurado
            or "intersecoes" in subpastas
        ):
            candidatas.add(atual)

def quantidade_encontrada(raiz):
    return sum(
        (raiz / cfg["relativo"]).is_file()
        for chave, cfg in CONFIGURACAO.items()
        if chave not in CAMINHOS_MANUAIS
    )

# 3. Escolher somente quando houver uma localização inequívoca.
necessitam_raiz = [
    chave for chave in CONFIGURACAO
    if chave not in CAMINHOS_MANUAIS
]

if necessitam_raiz:
    pontuacoes = {
        p: quantidade_encontrada(p)
        for p in candidatas
    }
    maior = max(pontuacoes.values(), default=0)

    if maior == 0:
        print("Locais pesquisados:")
        for p in locais:
            print(" -", p)

        raise FileNotFoundError(
            "Não encontrei os arquivos nas pastas disponíveis. "
            "No Colab, confira se a pasta Envio está em Meu Drive "
            "ou adicione um atalho para ela. Fora do Colab, "
            "disponibilize os arquivos no ambiente. "
            "Depois execute esta célula novamente."
        )

    melhores = [
        p for p, quantidade in pontuacoes.items()
        if quantidade == maior
    ]

    # Manter a configuração explícita quando ela já localiza os arquivos.
    if raiz_anterior in melhores:
        RAIZ = raiz_anterior
    elif len(melhores) == 1:
        RAIZ = melhores[0]
    else:
        print("Encontrei mais de uma pasta possível:")
        for p in sorted(melhores, key=str):
            print(" -", p)

        raise RuntimeError(
            "Para evitar misturar versões, defina RAIZ com "
            "um dos caminhos acima e execute novamente."
        )

print("RAIZ utilizada:", RAIZ)

# 4. Conferir cada arquivo e seu cabeçalho.
itens = []

for chave, cfg in CONFIGURACAO.items():
    p = Path(
        CAMINHOS_MANUAIS.get(chave, Path(RAIZ) / cfg["relativo"])
    )

    item = {
        "conjunto": chave,
        "encontrado": p.is_file(),
        "caminho": str(p),
    }

    if p.is_file():
        sep, cols = cabecalho(p)
        item.update({
            "tamanho_GB": round(p.stat().st_size / 1e9, 3),
            "registros_esperados": cfg["esperado"]["registros"],
            "colunas_omitidas": ", ".join(
                c for c in cols if geometrica(c)
            ),
        })

    itens.append(item)

display(pd.DataFrame(itens))

ausentes = [item for item in itens if not item["encontrado"]]

if ausentes:
    print("\nArquivos ainda não encontrados:")
    for item in ausentes:
        print(f"\n{item['conjunto']}:\n{item['caminho']}")

    raise FileNotFoundError(
        "A pasta foi localizada, mas faltam arquivos nos caminhos "
        "esperados. Ajuste CAMINHOS_MANUAIS para os itens acima."
    )

total_gb = sum(
    Path(item["caminho"]).stat().st_size for item in itens
) / 1e9

print(f"\nTodos os arquivos encontrados. Total: {total_gb:.2f} GB.")
print("Pode continuar para a próxima célula.")

RAIZ utilizada: /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios


,conjunto,encontrado,caminho,tamanho_GB,registros_esperados,colunas_omitidas
0,terras_indigenas,True,/content/drive/MyDrive/Colab Notebooks/fluxo/E...,0.075,660,"GT_GEOMETRIA, GEOMETRIA_TI_WKT, GEOMETRIA_INTE..."
1,quilombolas,True,/content/drive/MyDrive/Colab Notebooks/fluxo/E...,0.054,368,"GEOMETRIA_QUILOMBOLA_WKT, GEOMETRIA_INTERSECAO..."
2,florestas_publicas,True,/content/drive/MyDrive/Colab Notebooks/fluxo/E...,20.917,33984,"GT_GEOMETRIA, GEOMETRIA_FLORESTA_PUBLICA_WKT, ..."
3,assentamentos,True,/content/drive/MyDrive/Colab Notebooks/fluxo/E...,0.583,33044,"GT_GEOMETRIA, GEOMETRIA_ASSENTAMENTO_WKT, GEOM..."
4,embargos,True,/content/drive/MyDrive/Colab Notebooks/fluxo/E...,0.018,931,"GT_GEOMETRIA, GEOMETRIA_EMBARGO_WKT, GEOMETRIA..."
5,unidades_conservacao,True,/content/drive/MyDrive/Colab Notebooks/fluxo/E...,12.744,18617,"GT_GEOMETRIA, GEOMETRIA_UC_WKT, GEOMETRIA_INTE..."
6,car_suspenso,True,/content/drive/MyDrive/Colab Notebooks/fluxo/E...,0.002,530,"GT_GEOMETRIA, GEOMETRIA_CAR_WKT, GEOMETRIA_INT..."
7,embargos_auditados,True,/content/drive/MyDrive/Colab Notebooks/fluxo/E...,0.000,931,
8,embargos_prioritarios,True,/content/drive/MyDrive/Colab Notebooks/fluxo/E...,0.000,37,



Todos os arquivos encontrados. Total: 34.39 GB.
Pode continuar para a próxima célula.


## 4. Gerar as cópias leves e o pacote

Cada execução grava seus resultados em uma pasta nova. Falhas de leitura e divergências ficam registradas e impedem a criação do ZIP. A conferência de 37 registros verifica também se eles são exatamente o subconjunto temporal dos 931 registros, com os mesmos atributos.


In [6]:
PASTA_PRODUTO, ARQUIVO_ZIP = executar_produto(
    RAIZ, SAIDA_RAIZ, CONFIGURACAO, CAMINHOS_MANUAIS, TAMANHO_BLOCO
)


Saída: /content/drive/MyDrive/Produto_dissertacao_Cerrado/produto_20261002T021503_814295Z
Processando terras_indigenas (0.08 GB)...
Processando quilombolas (0.05 GB)...
Processando florestas_publicas (20.92 GB)...
  florestas_publicas: 1,000 registros lidos
  florestas_publicas: 2,000 registros lidos
  florestas_publicas: 3,000 registros lidos
  florestas_publicas: 4,000 registros lidos
  florestas_publicas: 5,000 registros lidos
  florestas_publicas: 6,000 registros lidos
  florestas_publicas: 7,000 registros lidos
  florestas_publicas: 8,000 registros lidos
  florestas_publicas: 9,000 registros lidos
  florestas_publicas: 10,000 registros lidos
  florestas_publicas: 11,000 registros lidos
  florestas_publicas: 12,000 registros lidos
  florestas_publicas: 13,000 registros lidos
  florestas_publicas: 14,000 registros lidos
  florestas_publicas: 15,000 registros lidos
  florestas_publicas: 16,000 registros lidos
  florestas_publicas: 17,000 registros lidos
  florestas_publicas: 18,000 r

## 5. Examinar o resultado

**Contagens conferidas** significa que os controles implementados passaram. Não significa que houve confirmação de irregularidades, revisão jurídica ou validação integral de todos os atributos. O dicionário pode conter definições pendentes, indicadas expressamente.

Não some as sete camadas. Não adicione os CSVs de 931 e 37 registros ao CSV geral de embargos: eles são visões do mesmo conjunto.


In [7]:
status = json.loads((PASTA_PRODUTO / 'status.json').read_text(encoding='utf-8'))
print(json.dumps(status, ensure_ascii=False, indent=2))
validacao = pd.read_csv(PASTA_PRODUTO / 'validacao.csv', sep=';', keep_default_na=False)
display(validacao)
pendencias = pd.read_csv(PASTA_PRODUTO / 'definicoes_pendentes.csv', sep=';', keep_default_na=False)
display(pendencias.head(30))
print('Confira todas as definições pendentes no CSV, não apenas as primeiras linhas exibidas.')
print('Pasta:', PASTA_PRODUTO)
print('ZIP:', ARQUIVO_ZIP or 'Não gerado: existem divergências.')


{
  "versao_notebook": "1.0",
  "gerado_utc": "20261002T021503_814295Z",
  "conjuntos_previstos": 9,
  "conjuntos_exportados": 9,
  "divergencias": 0,
  "falhas": [],
  "definicoes_pendentes": 303,
  "status": "CONTAGENS_CONFERIDAS_DOCUMENTACAO_PARA_REVISAO",
  "verificacao_administrativa": "não executada",
  "intersecoes_recalculadas": false
}


,conjunto,teste,esperado,calculado,confere
0,terras_indigenas,registros,660,660,True
1,terras_indigenas,glebas,554,554,True
2,terras_indigenas,operacoes,476,476,True
3,terras_indigenas,unidades,43,43,True
4,terras_indigenas,vazios_glebas,0,0,True
...,...,...,...,...,...
84,conferencia_temporal,mesmos_identificadores_do_recorte,True,True,True
85,conferencia_temporal,rotulos_prioritario_invalidos,0,0,True
86,conferencia_temporal,rotulo_prioritario_coerente,True,True,True
87,conferencia_temporal,rotulos_no_recorte,True,True,True


,conjunto,coluna,incluida_no_csv_leve,descricao,tipo_logico,unidade,situacao_definicao,vazios,registros,armazenamento
0,terras_indigenas,NU_INDICE,True,Atributo preservado literalmente. Consultar o ...,texto de origem,não confirmada,definição pendente,0,660,texto; valores e separadores decimais preservados
1,terras_indigenas,CD_CATEG_EMITENTE,True,Atributo preservado literalmente. Consultar o ...,texto de origem,não confirmada,definição pendente,0,660,texto; valores e separadores decimais preservados
2,terras_indigenas,CD_ESTADO,True,Atributo preservado literalmente. Consultar o ...,texto de origem,não confirmada,definição pendente,0,660,texto; valores e separadores decimais preservados
3,terras_indigenas,CD_PROGRAMA,True,Atributo preservado literalmente. Consultar o ...,texto de origem,não confirmada,definição pendente,0,660,texto; valores e separadores decimais preservados
4,terras_indigenas,VL_AREA_FINANC,True,Atributo preservado literalmente. Consultar o ...,texto de origem,não confirmada,definição pendente,0,660,texto; valores e separadores decimais preservados
5,terras_indigenas,CD_SUBPROGRAMA,True,Atributo preservado literalmente. Consultar o ...,texto de origem,não confirmada,definição pendente,420,660,texto; valores e separadores decimais preservados
6,terras_indigenas,VL_AREA_INFORMADA,True,Atributo preservado literalmente. Consultar o ...,texto de origem,não confirmada,definição pendente,0,660,texto; valores e separadores decimais preservados
7,terras_indigenas,DESC_PROGRAMA,True,Atributo preservado literalmente. Consultar o ...,texto de origem,não confirmada,definição pendente,0,660,texto; valores e separadores decimais preservados
8,terras_indigenas,DESC_CATEG_EMITENTE,True,Atributo preservado literalmente. Consultar o ...,texto de origem,não confirmada,definição pendente,0,660,texto; valores e separadores decimais preservados
9,terras_indigenas,DESC_SUBPROGRAMA,True,Atributo preservado literalmente. Consultar o ...,texto de origem,não confirmada,definição pendente,420,660,texto; valores e separadores decimais preservados


Confira todas as definições pendentes no CSV, não apenas as primeiras linhas exibidas.
Pasta: /content/drive/MyDrive/Produto_dissertacao_Cerrado/produto_20261002T021503_814295Z
ZIP: /content/drive/MyDrive/Produto_dissertacao_Cerrado/produto_20261002T021503_814295Z.zip


## 6. Download opcional

Os resultados já ficam no seu Drive. Execute a célula abaixo somente se também quiser baixar o ZIP no computador.


In [8]:
BAIXAR_ZIP = False  # mude para True para baixar
if BAIXAR_ZIP and ARQUIVO_ZIP is not None:
    from google.colab import files
    files.download(str(ARQUIVO_ZIP))
elif BAIXAR_ZIP:
    print('ZIP indisponível. Confira as divergências antes de entregar os arquivos.')


In [9]:
from pathlib import Path
import json
import pandas as pd

pasta = Path(
    "/content/drive/MyDrive/Produto_dissertacao_Cerrado/"
    "produto_20261002T021503_814295Z"
)

arquivo_status = pasta / "status.json"

if arquivo_status.exists():
    status = json.loads(arquivo_status.read_text(encoding="utf-8"))
    print("RESULTADO DO PROCESSAMENTO")
    print(json.dumps(status, ensure_ascii=False, indent=2))
else:
    print("status.json não encontrado: o processamento pode estar incompleto.")

arquivo_validacao = pasta / "validacao.csv"

if arquivo_validacao.exists():
    validacao = pd.read_csv(
        arquivo_validacao, sep=";", dtype=str, keep_default_na=False
    )
    divergencias = validacao[
        validacao["confere"].str.strip().str.lower() != "true"
    ]

    print(f"\nConferências registradas: {len(validacao)}")
    print(f"Divergências: {len(divergencias)}")

    if not divergencias.empty:
        display(divergencias)
else:
    print("\nArquivo de validação não encontrado.")

zip_path = pasta.with_suffix(".zip")
print("\nZIP gerado:", zip_path.is_file())

RESULTADO DO PROCESSAMENTO
{
  "versao_notebook": "1.0",
  "gerado_utc": "20261002T021503_814295Z",
  "conjuntos_previstos": 9,
  "conjuntos_exportados": 9,
  "divergencias": 0,
  "falhas": [],
  "definicoes_pendentes": 303,
  "status": "CONTAGENS_CONFERIDAS_DOCUMENTACAO_PARA_REVISAO",
  "verificacao_administrativa": "não executada",
  "intersecoes_recalculadas": false
}

Conferências registradas: 89
Divergências: 0

ZIP gerado: True


In [10]:
from google.colab import files
from pathlib import Path

arquivo = Path(
    "/content/drive/MyDrive/Produto_dissertacao_Cerrado/"
    "produto_20261002T021503_814295Z/definicoes_pendentes.csv"
)

if arquivo.is_file():
    files.download(str(arquivo))
else:
    print("Arquivo não encontrado nesse caminho.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>